# Lecture 3 — Connecting to APIs

This notebook demonstrates how to retrieve data from web APIs using Python's `requests`
library. APIs (Application Programming Interfaces) provide a standardised way to access
data from online services.

We use free, public APIs that do not require authentication to keep things simple in this notebook. Later in the exercises we will use an API that requires using a private, personal API key.

**Credits:**

Created by Sippo Rossi for the course Python Programming for Business Intelligence at Hanken.

## 1. What is an API?

An API is a mechanism that allows programs to communicate with each other. In practice,
you send a request to a URL and receive structured data (usually JSON) in return.

Many organisations offer APIs: X (Twitter), Yahoo Finance, government open data portals, etc.

**Important:** Many APIs require an API key, a unique identifier linked to your account.
API keys should be treated like passwords: never share them and never upload them to GitHub.

## 2. Making requests with `requests`

In [2]:
import requests
import pandas as pd

### 2.1 A simple GET request

We start with a free API that returns exchange rates. 

[ExchangeRate-API](https://www.exchangerate-api.com/docs/free) is a long-running service that provides currency conversion data for all major currencies. Its open-access endpoint requires no API key and no authentication. A GET request to `https://open.er-api.com/v6/latest/{base}` returns the latest exchange rates from the specified base currency to all supported currencies. The data refreshes once every 24 hours. The GET request retrieves data from the specified URL.


In [3]:
# Fetch latest EUR exchange rates from a free API
url = "https://open.er-api.com/v6/latest/EUR"
response = requests.get(url)

print("Status code:", response.status_code)
print("A status code of 200 means the request was successful")

Status code: 200
A status code of 200 means the request was successful


### 2.2 Working with JSON responses

Most APIs return data in JSON format, which maps naturally to Python dictionaries.

In [4]:
# Parse the JSON response
data = response.json()

# See what keys are available
print("Keys:", list(data.keys()))

Keys: ['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix', 'time_last_update_utc', 'time_next_update_unix', 'time_next_update_utc', 'time_eol_unix', 'base_code', 'rates']


In [5]:
# Access specific exchange rates
rates = data["rates"]
print(f"1 EUR = {rates['USD']} USD")
print(f"1 EUR = {rates['GBP']} GBP")
print(f"1 EUR = {rates['SEK']} SEK")

1 EUR = 1.149795 USD
1 EUR = 0.867789 GBP
1 EUR = 10.888389 SEK


## 3. Passing parameters to an API

Many APIs accept parameters that control what data is returned. These can be passed
as a dictionary.

In the next example, we use the Open-Meteo API.

[Open-Meteo](https://open-meteo.com/) is an open-source weather API that aggregates high-resolution forecast models from national weather services worldwide. It is free for non-commercial use, requires no API key, and returns data as JSON over a simple HTTP interface.

The `/v1/forecast` endpoint accepts a geographic coordinate and a list of desired weather variables, and returns an hourly forecast for up to 16 days (7 days by default). For Helsinki, the request would include the city's latitude and longitude as query parameters alongside the variables of interest (e.g. temperature, precipitation, wind speed).

```python
# The Open-Meteo weather API - get a 7-day forecast for Helsinki
url = "https://api.open-meteo.com/v1/forecast"
```

In [6]:
# The Open-Meteo weather API - get a 7-day forecast for Helsinki
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 60.17,
    "longitude": 24.94,
    "daily": "temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Helsinki"
}

response = requests.get(url, params=params)
weather = response.json()

print("Status code:", response.status_code)

Status code: 200


In [7]:
# Inspect the daily forecast data
daily = weather["daily"]

for i in range(len(daily["time"])):
    date = daily["time"][i]
    t_max = daily["temperature_2m_max"][i]
    t_min = daily["temperature_2m_min"][i]
    print(f"{date}: {t_min}°C - {t_max}°C")

2026-03-30: 0.8°C - 3.7°C
2026-03-31: 1.2°C - 8.5°C
2026-04-01: 0.7°C - 5.2°C
2026-04-02: 0.2°C - 5.0°C
2026-04-03: 2.2°C - 7.1°C
2026-04-04: -0.1°C - 4.8°C
2026-04-05: 1.8°C - 5.6°C


## 4. Converting API data to a DataFrame

Since API responses are typically dictionaries or lists of dictionaries, they can be
converted to DataFrames conveniently.

In [8]:
# Convert the weather data to a DataFrame
df_weather = pd.DataFrame({
    "date": daily["time"],
    "temp_max": daily["temperature_2m_max"],
    "temp_min": daily["temperature_2m_min"]
})

df_weather

,date,temp_max,temp_min
0,2026-03-30,3.7,0.8
1,2026-03-31,8.5,1.2
2,2026-04-01,5.2,0.7
3,2026-04-02,5.0,0.2
4,2026-04-03,7.1,2.2
5,2026-04-04,4.8,-0.1
6,2026-04-05,5.6,1.8


In [9]:
# Convert the exchange rates to a DataFrame
df_rates = pd.DataFrame(
    list(rates.items()),
    columns=["currency", "rate"]
)

df_rates.head(10)

,currency,rate
0,EUR,1.000000
1,AED,4.222636
2,AFN,72.993432
3,ALL,96.070456
4,AMD,434.106510
5,ANG,2.058140
6,AOA,1070.368744
7,ARS,1582.236656
8,AUD,1.676336
9,AWG,2.058140


## 5. Error handling

API requests can fail for many reasons: the server is down, the URL is wrong, or your
internet connection is lost. It is good practice to check the status code before
processing the response.

In [10]:
# Example: requesting a URL that does not exist
response = requests.get("https://api.open-meteo.com/v1/nonexistent")
print("Status code:", response.status_code)

if response.status_code == 200:
    print("Success")
else:
    print("Request failed")

Status code: 404
Request failed


Common HTTP status codes:

| Code | Meaning |
|---|---|
| 200 | OK - the request was successful |
| 400 | Bad Request - something was wrong with your request |
| 401 | Unauthorised - you need an API key |
| 404 | Not Found - the URL does not exist |
| 429 | Too Many Requests - you have exceeded the rate limit |

## 6. A note on API keys

Many APIs require an API key for authentication. When using API keys:

1. **Never hardcode them** in your notebook or script.
2. **Never upload them** to GitHub (add config files to `.gitignore`).
3. Store them in a separate file (e.g. `config.json`) or as environment variables. This will be practiced during this week's exercise session.

Example pattern for loading a key from a file:

```python
import json

with open("config.json") as f:
    config = json.load(f)

api_key = config["api_key"]
```